# Attention 메커니즘 완전 정복 - 실습 코드 1: Self-Attention 구현 (PyTorch)

- Tutorial ID: `expand-attention-mechanism`
- Tutorial: Attention 메커니즘 완전 정복
- Section ID: `expand-attention-mechanism-code-1`
- Section: 실습 코드 1: Self-Attention 구현 (PyTorch)

> 이 노트북은 Self-Attention을 **처음 배우는 분**을 위한 실습 자료입니다. 새로운 개념이 나올 때마다 바로 설명을 붙이고, 아주 작은 숫자로 직접 계산을 확인한 뒤에 실제 크기로 넘어가는 순서로 구성했습니다.

In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: Self-Attention 구현 (PyTorch)
#
# 이 노트북은 "정답 코드를 한 번 실행해보고 끝"내는 용도가 아니라,
# Self-Attention의 수식이 실제 텐서 연산으로 바뀌는 과정을
# 한 줄씩, 작은 숫자로 직접 추적해보기 위한 실습 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고, 어떻게 attention score로 이어지는지 추적한다.
#   2) score가 scaling(나누기 sqrt(d_k))과 softmax를 거쳐
#      "합이 1인 확률(가중치)"로 바뀌는 과정을 직접 숫자로 확인한다.
#   3) 미래 토큰을 마스킹(-1e9)했을 때, softmax 이후 그 위치의 확률이
#      0에 가까워지는 것을 직접 확인한다.
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model, d_k)를 먼저 확인합니다.
#   2) 입력 x가 어떤 shape으로 만들어지는지 봅니다.
#   3) W_q/W_k/W_v가 x를 어떤 차원(공간)으로 투영하는지 확인합니다.
#   4) bmm, softmax, masked_fill 등 핵심 연산 직후의 shape와 값을
#      출력(print)으로 직접 검증합니다.
#   5) seed, d_model, d_k, seq_len, mask 유무 등을 바꿔가며
#      결과가 어떻게 달라지는지 실험해봅니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 "shape이 어떻게 바뀌는지"와
#     "정보가 어느 방향으로 섞이는지"를 보세요.
#   - 이 노트북은 PyTorch만 설치되어 있으면 실행됩니다.
#     로컬에 PyTorch가 없다면 Colab(colab.research.google.com)에서
#     이 파일을 열어 실행하는 것을 권장합니다.

## 1. Self-Attention, 왜 필요할까요?

문장 하나를 예로 들어볼게요.

> "그 동물은 길을 건너지 않았다. 왜냐하면 **그것**이 너무 피곤했기 때문이다."

이 문장에서 **"그것"**이 가리키는 대상은 "동물"일까요, "길"일까요? 우리는 문맥을 보고 바로 "동물"이라는 걸 알 수 있습니다.

모델도 "그것"이라는 단어를 처리할 때, 문장 속 **다른 모든 단어를 한 번씩 둘러보면서** "동물"과 "길" 중 어디에 더 집중해야 할지를 점수로 계산합니다. 이렇게 **한 문장 안에서 단어들끼리 서로 얼마나 관련 있는지 점수를 매기고, 그 점수에 따라 정보를 섞어주는 것**이 Self-Attention입니다.

- **Self (자기 자신)**: 다른 문장이 아니라 **자기 자신을 이루는 단어들끼리** 관계를 계산하기 때문에 붙은 이름입니다.
- **Attention (주의)**: 모든 단어에 똑같이 신경 쓰는 게 아니라, **관련 있는 단어에 더 많은 주의(가중치)를 주기** 때문에 붙은 이름입니다.

이번 실습에서는 이 과정을 **Query(질의), Key(키), Value(값)** 라는 세 개의 벡터로 어떻게 계산하는지 코드로 직접 따라가 봅니다.

## 2. Query, Key, Value를 도서관에 비유해서 이해하기

Self-Attention의 핵심은 **Q(Query), K(Key), V(Value)** 세 가지입니다. 도서관에서 책을 찾는 상황에 비유해 보겠습니다.

| 개념 | 도서관 비유 | 의미 |
|---|---|---|
| **Query (질의)** | 내가 사서에게 묻는 질문: "AI 관련 책 있나요?" | "나는 지금 어떤 정보를 찾고 있다"를 표현한 벡터 |
| **Key (키)** | 책마다 붙어 있는 분류표/라벨: "이 책은 AI 책입니다" | "나는 이런 정보를 갖고 있다"를 표현한 벡터 |
| **Value (값)** | 책의 실제 내용 | Query와 Key가 잘 맞을 때 실제로 가져올 정보 |

문장 속 단어 하나하나는 **Query 역할(나는 무엇을 찾고 싶은가)**, **Key 역할(나는 어떤 정보를 갖고 있는가)**, **Value 역할(내가 가진 실제 정보)**을 동시에 합니다. 즉 모든 단어가 서로에게 "내 질문(Query)과 네 라벨(Key)이 얼마나 잘 맞니?"를 물어보고, 잘 맞을수록 그 단어의 Value를 더 많이 가져오는 것이죠.

계산 순서는 다음과 같습니다.

1. 입력 `x`에 서로 다른 세 개의 선형변환(Linear layer)을 적용해서 `Q`, `K`, `V`를 만든다.
2. `Q`와 `K`를 내적(dot product)해서 "얼마나 관련 있는지" 점수(score)를 구한다.
3. 점수를 `sqrt(d_k)`로 나눠 스케일을 맞춘다 (이유는 아래에서 설명합니다).
4. `softmax`로 점수를 "합이 1인 확률"로 바꾼다 → 이것이 **attention weight(가중치)**.
5. 가중치로 `V`들을 가중합(weighted sum)해서 최종 출력을 만든다.

이 5단계를 아래에서 아주 작은 숫자로 하나씩 직접 확인해보겠습니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 실행할 때마다 같은 난수가 나오도록 시드(seed)를 고정합니다.
# 이렇게 해두면 이 노트북을 다시 실행해도 항상 같은 숫자가 나오므로,
# "내가 보는 숫자가 설명과 다른가?" 하는 혼란 없이 결과를 비교할 수 있습니다.
torch.manual_seed(42)

print("PyTorch 버전:", torch.__version__)

## 3. 아주 작은 숫자로 먼저 손으로 확인해보기

처음부터 `d_model=512` 같은 큰 차원을 보면 숫자가 너무 많아서 흐름을 따라가기 어렵습니다. 그래서 먼저 **아주 작은 예시**로 Q/K/V → score → softmax → output까지 전체 흐름을 직접 눈으로 확인할 수 있을 정도로 만들어 보겠습니다.

- 문장: "나는 밥을 먹는다" → 토큰(단어) 3개로 가정 (`나는`, `밥을`, `먹는다`)
- 각 단어는 4개의 숫자(4차원 벡터)로 표현 (`d_model=4`)
- `batch_size=1` (문장 1개만 사용)

실제 모델에서는 단어를 임베딩(embedding)이라는 과정을 통해 벡터로 바꾸지만, 여기서는 설명을 위해 **이미 벡터로 변환되어 있다고 가정**하고 랜덤 숫자를 사용합니다.

In [ ]:
# (batch_size=1, seq_len=3, d_model=4) 모양의 아주 작은 입력을 만듭니다.
# "단어 3개가 각각 4개의 숫자(차원)로 표현되어 있다"고 생각하면 됩니다.
batch_size = 1
seq_len = 3       # "나는", "밥을", "먹는다" -> 단어(토큰) 3개
d_model = 4       # 단어 하나를 표현하는 벡터의 차원 수

x_tiny = torch.randn(batch_size, seq_len, d_model)

print("x_tiny.shape:", x_tiny.shape)
print(x_tiny)

### 3-1. Q, K, V 만들기 — `nn.Linear`는 무엇을 하는 걸까?

`nn.Linear(d_model, d_k)`는 `d_model`차원 벡터를 입력받아 **다른 차원(`d_k`)의 벡터로 바꿔주는 학습 가능한 변환**입니다. 수식으로 쓰면 다음과 같습니다.

```
출력 = 입력 @ W^T + b
```

여기서 `W`(가중치 행렬)와 `b`(편향)는 학습 과정에서 점점 더 좋은 값으로 업데이트되는 파라미터입니다. 지금은 학습을 시작하기 전이라 `W`와 `b`가 랜덤한 값이라는 점만 기억하면 됩니다.

같은 입력 `x`에 **서로 다른 세 개의 Linear layer**(`W_q`, `W_k`, `W_v`)를 적용하면, 같은 단어라도 "Query로 쓰일 때", "Key로 쓰일 때", "Value로 쓰일 때" 각각 다른 벡터가 됩니다. 단어 자기 자신을 그대로 비교하는 대신 **"질문하는 나"와 "답하는 나"를 다르게 표현**할 수 있어야, 모델이 더 풍부한 관계를 학습할 수 있기 때문입니다.

In [ ]:
d_k = 3  # Q, K, V를 투영할 차원 (꼭 d_model과 같을 필요는 없습니다)

# 실제 모델에서는 이 Linear layer들을 nn.Module 클래스 안에서 정의하지만,
# 지금은 동작 원리를 보기 위해 일단 따로 떼어내어 만들어봅니다.
W_q = nn.Linear(d_model, d_k)
W_k = nn.Linear(d_model, d_k)
W_v = nn.Linear(d_model, d_k)

Q = W_q(x_tiny)  # (1, 3, 3) : "질문" 역할의 벡터
K = W_k(x_tiny)  # (1, 3, 3) : "라벨" 역할의 벡터
V = W_v(x_tiny)  # (1, 3, 3) : "실제 내용" 역할의 벡터

print("Q.shape:", Q.shape)
print("K.shape:", K.shape)
print("V.shape:", V.shape)
print()
print("Q:\n", Q)

### 3-2. Attention Score 계산하기 — `Q @ K^T`

이제 "단어 i의 Query가 단어 j의 Key와 얼마나 잘 맞는가"를 계산합니다. 이건 두 벡터의 **내적(dot product)**으로 구합니다. 내적값이 클수록 두 벡터의 방향이 비슷하다(=관련이 깊다)는 뜻입니다.

`Q`의 모양은 `(batch, seq_len, d_k)`이고, 모든 단어 쌍의 점수를 한 번에 구하려면 `Q`와 `K`를 전치(transpose)한 행렬을 곱하면 됩니다.

```
scores = Q @ K^T
       # (batch, seq_len, d_k) @ (batch, d_k, seq_len) = (batch, seq_len, seq_len)
```

결과로 나오는 `scores`는 `(seq_len, seq_len)` 모양의 행렬이고, `scores[i][j]`는 **"단어 i가 단어 j를 얼마나 봐야 하는가"**의 점수(아직 확률은 아닙니다)를 의미합니다.

코드에서 쓰는 두 함수를 짚고 넘어갈게요.

- `K.transpose(1, 2)` : `(batch, seq_len, d_k)` → `(batch, d_k, seq_len)`로, 1번째와 2번째 차원(0번째인 batch는 그대로 두고)을 맞바꿉니다.
- `torch.bmm(A, B)` : "batch matrix multiplication"의 줄임말로, `batch` 차원은 그대로 유지한 채 나머지 2개 차원끼리만 행렬곱을 해줍니다. (참고: 최신 PyTorch 코드에서는 `torch.bmm` 대신 더 범용적인 `A @ B`(=`torch.matmul`)를 쓰기도 하는데, 3차원 텐서에서는 둘이 동일하게 동작합니다.)

In [ ]:
# torch.bmm = batch matrix multiplication. (batch 차원은 유지한 채, 나머지 2D 행렬끼리 행렬곱)
# K.transpose(1, 2) : (batch, seq_len, d_k) -> (batch, d_k, seq_len)로 마지막 두 축을 swap
scores = torch.bmm(Q, K.transpose(1, 2))

print("scores.shape:", scores.shape)  # (1, 3, 3) : 단어 3개 x 단어 3개 = 모든 쌍의 점수
print(scores)

### 3-3. 왜 `sqrt(d_k)`로 나눠줄까? (Scaling)

`d_k`(벡터 차원)가 커질수록 내적값의 절댓값도 함께 커지는 경향이 있습니다. 점수가 너무 크면 바로 다음 단계인 `softmax`를 통과할 때 **한쪽 값만 1에 가깝게, 나머지는 거의 0에 가깝게** 만들어버립니다. 이렇게 되면 학습 중 기울기(gradient)가 거의 사라져서 모델이 잘 학습되지 않는 문제가 생깁니다.

이를 막기 위해 점수를 `sqrt(d_k)`로 나눠 값의 크기(분산)를 일정하게 맞춰줍니다. 이것이 바로 **"Scaled" Dot-Product Attention**이라는 이름의 유래입니다.

```
scores = (Q @ K^T) / sqrt(d_k)
```

`d_k`가 클수록 나누는 값도 커지므로, 점수가 너무 한쪽으로 쏠리는 것을 막아줍니다.

In [ ]:
scale = d_k ** 0.5  # sqrt(d_k)
print(f"d_k = {d_k}, scale(=sqrt(d_k)) = {scale:.4f}")

scaled_scores = scores / scale

print("\n나누기 전 scores:\n", scores)
print("\n나누기 후 scaled_scores:\n", scaled_scores)

### 3-4. Softmax로 "확률"로 바꾸기

`scaled_scores`는 아직 임의의 실수(점수)일 뿐, "확률"이 아닙니다. `softmax` 함수는 한 행(row)의 숫자들을 **모두 0~1 사이의 값으로 바꾸면서, 그 행의 합이 정확히 1이 되도록** 만들어줍니다.

```
softmax(z_i) = exp(z_i) / sum_j( exp(z_j) )
```

즉 `weights[i][j]`는 **"단어 i가 출력을 만들 때, 단어 j의 정보를 몇 % 반영할지"**를 나타내는 값이 됩니다.

`F.softmax(scores, dim=-1)`에서 `dim=-1`(마지막 차원)을 지정하는 이유는, **각 단어(행) 안에서** 다른 모든 단어에 대한 점수들끼리 합이 1이 되도록 정규화하고 싶기 때문입니다. 만약 반대 방향(열 방향)으로 정규화하면 "이 단어가 다른 단어들에게 얼마나 주목받는가"라는, 우리가 원하는 것과는 다른 의미가 되어버립니다.

In [ ]:
weights = F.softmax(scaled_scores, dim=-1)

print("weights:\n", weights)

# 검증: 각 행(각 단어)의 가중치 합이 정확히 1이 되는지 확인합니다.
print("\n각 행의 합 (전부 1이어야 정상):")
print(weights.sum(dim=-1))

### 3-5. 가중치로 Value를 섞어 최종 출력 만들기

마지막으로, 각 단어의 출력은 **모든 단어의 Value를 가중치(weights)만큼 섞어서** 만듭니다.

```
output = weights @ V
       # (batch, seq, seq) @ (batch, seq, d_k) = (batch, seq, d_k)
```

예를 들어 1번째 단어의 출력은 `weights[0]`(1번째 단어가 다른 모든 단어에 주는 가중치)과 `V`의 가중합입니다. 만약 `weights[0] = [0.7, 0.2, 0.1]`이라면, 1번째 단어의 출력은 `0.7 * V[0] + 0.2 * V[1] + 0.1 * V[2]`가 됩니다 — 1번째 단어 자신의 정보를 70%, 2번째 단어 정보를 20%, 3번째 단어 정보를 10% 반영한 셈입니다.

In [ ]:
output = torch.bmm(weights, V)

print("output.shape:", output.shape)  # (1, 3, 3) : 단어 3개, 각각 d_k=3 차원
print(output)

# 직접 손으로 확인해보기: 1번째 단어("나는")의 출력이
# weights[0,0]을 가중치로 V를 가중합한 값과 정확히 같은지 검증합니다.
manual_output_0 = (weights[0, 0, 0] * V[0, 0]
                    + weights[0, 0, 1] * V[0, 1]
                    + weights[0, 0, 2] * V[0, 2])

print("\n코드로 계산한 output[0,0]   :", output[0, 0])
print("직접 가중합으로 계산한 값   :", manual_output_0)
print("두 값이 (거의) 같은가?      :", torch.allclose(output[0, 0], manual_output_0, atol=1e-6))

## 4. 지금까지의 과정을 `nn.Module` 클래스로 정리하기

방금 한 단계씩 진행한 과정을 정리하면 다음과 같습니다.

1. `Q = W_q(x)`, `K = W_k(x)`, `V = W_v(x)`
2. `scores = (Q @ K^T) / sqrt(d_k)`
3. (필요하다면) 마스킹
4. `weights = softmax(scores)`
5. `output = weights @ V`

이 5단계를 매번 따로 적지 않고 재사용하기 쉽도록, PyTorch의 `nn.Module`을 상속받는 클래스로 묶어보겠습니다. 클래스 구조가 낯설다면 다음과 같이 이해하면 됩니다.

- **`__init__`**: "이 모듈이 어떤 학습 가능한 부품(Linear layer 등)을 가지고 있는지" 정의하는 곳. 모델을 만들 때 한 번만 실행됩니다.
- **`forward`**: "입력이 들어왔을 때 그 부품들을 어떤 순서로 사용할지" 정의하는 곳. 모델을 호출(`model(x)`)할 때마다 실행됩니다.

아래 코드는 위에서 손으로 진행한 단계와 **완전히 동일한 계산**을 하지만, mask를 처리하는 부분이 하나 추가되어 있습니다. mask는 바로 다음 6번 섹션에서 자세히 다룹니다.

In [ ]:
class SelfAttention(nn.Module):
    """
    가장 기본적인 형태의 Self-Attention (Single-Head, 헤드가 1개인 버전).

    입력 x       : (batch_size, seq_len, d_model)
    출력 output  : (batch_size, seq_len, d_k)
    출력 weights : (batch_size, seq_len, seq_len)  -> "누가 누구를 얼마나 봤는지"
    """

    def __init__(self, d_model, d_k):
        # nn.Module을 상속받는 클래스는 항상 super().__init__()을 가장 먼저 호출해야
        # PyTorch 내부적으로 파라미터(W_q, W_k, W_v 등)를 추적하는 기능이 정상 동작합니다.
        super().__init__()

        # x(d_model 차원)를 Q, K, V(d_k 차원)로 각각 투영하는 학습 가능한 선형변환입니다.
        # W_q, W_k, W_v는 서로 독립적인 파라미터이며, 학습을 거치면서 각자 다른 값으로 수렴합니다.
        self.W_q = nn.Linear(d_model, d_k)
        self.W_k = nn.Linear(d_model, d_k)
        self.W_v = nn.Linear(d_model, d_k)

        # 점수를 나눠줄 스케일 값(sqrt(d_k))을 미리 계산해서 저장해 둡니다.
        self.scale = d_k ** 0.5

    def forward(self, x, mask=None):
        # x: (batch, seq, d_model)
        Q = self.W_q(x)  # (batch, seq, d_k) : "나는 무엇을 찾고 있는가"
        K = self.W_k(x)  # (batch, seq, d_k) : "나는 어떤 정보를 가지고 있는가"
        V = self.W_v(x)  # (batch, seq, d_k) : "내가 실제로 가진 정보(내용)"

        # --- 1) Attention score 계산: Q와 K가 얼마나 잘 맞는지 ---
        # K.transpose(1, 2) : (batch, seq, d_k) -> (batch, d_k, seq)
        # torch.bmm        : 배치 차원(batch)은 유지한 채 나머지 2D 행렬끼리 행렬곱
        # scores           : (batch, seq, seq) -> scores[b, i, j] = i번째 단어가 j번째 단어를 보는 점수
        scores = torch.bmm(Q, K.transpose(1, 2)) / self.scale

        # --- 2) (선택) 마스킹: 특정 위치를 "보지 못하게" 막기 ---
        # mask에서 값이 0인 위치는 절대 못 보게 아주 작은 값(-1e9)으로 채워버립니다.
        # 이렇게 하면 잠시 후 softmax를 통과했을 때 해당 위치의 확률이 0에 가까워집니다.
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        # --- 3) Softmax로 점수를 확률(가중치)로 변환 ---
        # dim=-1 : 마지막 차원(= 각 단어가 다른 모든 단어를 보는 점수들) 기준으로 정규화
        # 즉 weights[b, i, :].sum() == 1 이 되도록 만듭니다.
        weights = F.softmax(scores, dim=-1)

        # --- 4) 가중치로 V를 가중합 -> 최종 출력 ---
        # (batch, seq, seq) @ (batch, seq, d_k) = (batch, seq, d_k)
        output = torch.bmm(weights, V)

        return output, weights

## 5. 클래스가 손으로 한 계산과 똑같이 동작하는지 확인

클래스 안의 `W_q`, `W_k`, `W_v`는 새로 만들 때마다 랜덤하게 초기화됩니다. 위에서 손으로 계산할 때 썼던 가중치와 **똑같은 값**을 복사해 넣은 뒤, 같은 입력(`x_tiny`)을 넣었을 때 결과가 정확히 같은지 비교해보겠습니다.

> 참고: 실제로 모델을 학습시킬 때는 이런 "가중치 복사" 작업이 전혀 필요 없습니다. 지금은 어디까지나 "클래스가 제대로 동작하는지" 검증하기 위한 코드입니다.

In [ ]:
# 새 SelfAttention 객체를 만들고, 위에서 손으로 계산할 때 썼던 것과
# 똑같은 W_q / W_k / W_v 가중치를 복사해 넣습니다.
attn_tiny = SelfAttention(d_model=d_model, d_k=d_k)

with torch.no_grad():  # 가중치를 복사하는 동안에는 기울기 계산이 필요 없습니다.
    attn_tiny.W_q.weight.copy_(W_q.weight)
    attn_tiny.W_q.bias.copy_(W_q.bias)
    attn_tiny.W_k.weight.copy_(W_k.weight)
    attn_tiny.W_k.bias.copy_(W_k.bias)
    attn_tiny.W_v.weight.copy_(W_v.weight)
    attn_tiny.W_v.bias.copy_(W_v.bias)

class_output, class_weights = attn_tiny(x_tiny)

print("직접 단계별로 계산한 output:\n", output)
print("\nSelfAttention 클래스로 계산한 output:\n", class_output)

print("\n두 결과가 (거의) 같은가?", torch.allclose(output, class_output, atol=1e-6))

## 6. 마스킹(Masking)이 필요한 이유

언어 모델이 **다음 단어를 예측**하는 상황을 생각해봅시다. "나는 밥을 ___"의 빈칸을 예측할 때, 모델은 "나는", "밥을"까지만 보고 다음 단어를 맞혀야 합니다. 만약 정답인 "먹는다"까지 미리 들여다볼 수 있다면 그건 반칙이고, 실제로 새 문장을 한 단어씩 생성해야 하는 상황에서는 애초에 불가능한 일이기도 합니다.

그래서 **자기보다 뒤(미래)에 있는 단어는 보지 못하게 점수를 강제로 -∞에 가깝게 만들어주는 장치**가 필요한데, 이를 **causal mask(인과적 마스크)** 또는 **look-ahead mask**라고 부릅니다.

> 참고: 마스킹에는 이 외에도 길이가 서로 다른 문장들을 하나의 batch로 묶을 때, 의미 없이 채워 넣은 padding 토큰을 무시하기 위한 **padding mask**도 있습니다. 이번 실습에서는 가장 핵심이 되는 causal mask를 직접 만들어봅니다.

causal mask는 보통 다음과 같은 **하삼각행렬(lower triangular matrix)** 형태입니다 (`seq_len=4`인 경우).

```
                   볼 수 있는 위치 (1=보임, 0=못 봄)
                   j=0   j=1   j=2   j=3
i=0 (1번째 단어)    [ 1     0     0     0 ]
i=1 (2번째 단어)    [ 1     1     0     0 ]
i=2 (3번째 단어)    [ 1     1     1     0 ]
i=3 (4번째 단어)    [ 1     1     1     1 ]
```

`i`번째 단어(행)는 자기 자신을 포함해서 **자기 이전(왼쪽, 과거)까지만** 볼 수 있고, 그 뒤(오른쪽, 미래)는 모두 0이라 못 봅니다.

코드에서는 `scores.masked_fill(mask == 0, -1e9)`로, mask가 0인 자리의 score를 아주 작은 음수(`-1e9`, 사실상 음의 무한대)로 바꿔버립니다. `softmax`를 통과한 뒤에는 `exp(-1e9) ≈ 0`이기 때문에, 그 자리의 확률은 사실상 0이 됩니다.

In [ ]:
def make_causal_mask(seq_len):
    """
    (seq_len, seq_len) 크기의 하삼각행렬 마스크를 만듭니다.
    대각선을 포함한 왼쪽 아래(과거+현재)는 1, 오른쪽 위(미래)는 0입니다.
    """
    return torch.tril(torch.ones(seq_len, seq_len))


seq_len_demo = 4
causal_mask = make_causal_mask(seq_len_demo)

print("causal_mask (1=볼 수 있음, 0=못 봄):")
print(causal_mask)

In [ ]:
torch.manual_seed(0)  # 이 섹션만 따로 실행해도 같은 결과가 나오도록 다시 시드를 고정합니다.

d_model_demo = 8
d_k_demo = 4

attn_causal = SelfAttention(d_model=d_model_demo, d_k=d_k_demo)
x_demo = torch.randn(1, seq_len_demo, d_model_demo)  # batch=1, seq=4

# 먼저 mask 없이 실행 -> 모든 단어가 다른 모든 단어를 자유롭게 봅니다.
_, weights_no_mask = attn_causal(x_demo, mask=None)

print("[마스킹 없음] weights:")
print(weights_no_mask[0])  # batch 차원 0번째만 출력

In [ ]:
# 이번엔 같은 모델, 같은 입력으로 causal_mask만 추가해서 다시 실행합니다.
_, weights_with_mask = attn_causal(x_demo, mask=causal_mask)

print("[causal mask 적용] weights:")
print(weights_with_mask[0])

print("\n각 행의 합 (마스킹을 해도 softmax이므로 여전히 1이어야 함):")
print(weights_with_mask[0].sum(dim=-1))

## 7. 마스킹 전후 비교 + 시각화

두 결과를 비교해보면 다음과 같은 패턴을 확인할 수 있습니다.

- **마스킹 없음**: 모든 행에서 4개 위치 전부에 0이 아닌 가중치가 퍼져 있습니다.
- **마스킹 적용**: `i`번째 행(단어)은 `j <= i`인 위치에만 0이 아닌 가중치를 가지고, `j > i`(미래)인 위치는 0에 매우 가깝습니다.
  - 특히 1번째 행(`i=0`)은 비교할 다른 후보가 아예 없으므로 가중치가 정확히 `[1, 0, 0, 0]`에 가깝게 나옵니다.
  - 마지막 행(`i=3`)은 마스킹 여부와 관계없이 두 결과가 동일합니다 — 마지막 단어는 어차피 모든 위치를 다 볼 수 있기 때문입니다.

숫자만 보면 패턴을 알아채기 어려울 수 있으니, 히트맵(heatmap)으로 그려서 한눈에 비교해보겠습니다. 색이 밝을수록(노란색에 가까울수록) 가중치가 크다는 뜻입니다.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

im0 = axes[0].imshow(weights_no_mask[0].detach().numpy(), cmap="viridis", vmin=0, vmax=1)
axes[0].set_title("마스킹 없음")
axes[0].set_xlabel("Key 위치 (j)")
axes[0].set_ylabel("Query 위치 (i)")
axes[0].set_xticks(range(seq_len_demo))
axes[0].set_yticks(range(seq_len_demo))
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(weights_with_mask[0].detach().numpy(), cmap="viridis", vmin=0, vmax=1)
axes[1].set_title("Causal mask 적용")
axes[1].set_xlabel("Key 위치 (j)")
axes[1].set_ylabel("Query 위치 (i)")
axes[1].set_xticks(range(seq_len_demo))
axes[1].set_yticks(range(seq_len_demo))
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.suptitle("Attention Weight 히트맵 (밝을수록 가중치가 큼)")
plt.tight_layout()
plt.show()

## 8. 실제 논문/모델에서 쓰는 크기로 실행해보기

지금까지는 손으로 따라갈 수 있도록 아주 작은 숫자(`d_model=4`, `seq_len=3` 등)를 사용했습니다. 이번에는 Transformer 논문에서 자주 등장하는 크기인 `d_model=512`로 똑같은 코드를 실행해서, **shape이 우리가 예상한 대로 나오는지** 확인해봅니다.

- `batch_size=2` : 문장 2개를 동시에 처리
- `seq_len=10` : 문장 하나당 단어(토큰) 10개
- `d_model=512` : 단어 하나를 표현하는 임베딩 차원
- `d_k=64` : Q/K/V를 투영할 차원

> 참고: `d_k=64`는 실제 Transformer 구현에서 자주 쓰이는 값입니다. 원논문에서는 `d_model=512`를 8개로 쪼개 각각 64차원씩 사용하는 **Multi-Head Attention**을 사용하는데, 이 부분은 다음 실습 코드에서 다룹니다. 지금은 "head가 1개뿐인 버전"이라고 생각하면 됩니다.

In [ ]:
# 사용 예시: 실제 Transformer 논문과 비슷한 크기로 실행해보기
attn = SelfAttention(d_model=512, d_k=64)

# batch=2 (문장 2개), seq=10 (단어 10개), d_model=512 (임베딩 차원)
x = torch.randn(2, 10, 512)

out, weights = attn(x)

print(f"입력 x       : {tuple(x.shape)}  (batch, seq_len, d_model)")
print(f"출력 output  : {tuple(out.shape)}  (batch, seq_len, d_k)")
print(f"출력 weights : {tuple(weights.shape)}  (batch, seq_len, seq_len)")

# weights는 "확률"이므로, 각 단어(행)에 대한 가중치 합은 항상 1이어야 합니다.
print("\n각 행의 합이 1에 가까운지 확인 (batch 0, 앞 3개 단어만):")
print(weights[0, :3].sum(dim=-1))

## 9. 정리

이번 실습에서 확인한 내용을 정리하면 다음과 같습니다.

1. **Self-Attention**은 한 문장 안의 단어들이 서로를 얼마나 참고해야 하는지 점수를 매기고, 그 점수에 따라 정보를 섞는 메커니즘입니다.
2. 입력 `x`를 세 개의 서로 다른 Linear layer에 통과시켜 **Q(질의), K(키), V(값)**를 만듭니다.
3. `Q @ K^T`로 모든 단어 쌍의 점수를 구하고, `sqrt(d_k)`로 나눠 값이 너무 커지지 않게 조정합니다 (Scaled Dot-Product).
4. `softmax`로 점수를 "합이 1인 확률(가중치)"로 바꿉니다.
5. 가중치로 `V`를 가중합하면 최종 출력이 됩니다.
6. **마스킹**을 사용하면 특정 위치(예: 미래 단어)를 보지 못하게 점수를 `-inf`에 가깝게 만들 수 있습니다.

### 확인 문제 (스스로 풀어보기)

아래 질문에 코드를 직접 수정해보면서 답해보세요.

1. `d_k`를 64에서 4로 바꾸면 `weights`의 shape이 바뀔까요, `output`의 shape이 바뀔까요? 직접 바꿔서 확인해보세요.
2. Scaling(나누기 `sqrt(d_k)`) 코드를 잠시 빼고 다시 실행하면 `weights` 값이 어떻게 달라지나요? (한쪽 단어로 확률이 더 심하게 쏠리는지 비교해보세요.)
3. `make_causal_mask(6)`을 직접 출력해서, `seq_len=6`일 때도 패턴이 예상한 대로 하삼각행렬인지 확인해보세요.

다음 실습 코드에서는 이 Self-Attention을 여러 개 동시에 사용하는 **Multi-Head Attention**을 다뤄보겠습니다.